# 機能テスト 1: Entity Graph(エンティティ グラフ)

## サマリー

- Entity Graph を**作る主体は、Foundry IQ サービス自体**です。ナレッジ ソースの `graphExtraction` を有効にすると、サービス内のマネージドな取り込みパイプラインが、チャンク化、ベクトル化、トリプル(主語・述語・目的語)の抽出を行います。プラグインの Python(`entity-graph-build`)は、定義 JSON を REST で送って状態を見張るだけです。このノートブックは、SDK から直接呼び出して、**サービスの機能をテストします**。
- 入力は、同じフォルダーにある 2 つのテキスト(`近藤寺 - Hanapedia.txt`、`花ケ咲神社 - Hanapedia.txt`)です。他の機能(Topic Graph、オントロジー生成)には依存しません。既定では、このノートブック内の小さな JSON-LD を使います。外部 JSON-LD の指定も任意で可能ですが、機能テスト 3 の実行は必須ではありません。
- 手順は、プラグインの「オントロジーを後から接続する」と同じです。**ナレッジ ソースをオントロジーなしで作り**(チャンク化とベクトル化だけ)、**あとからオントロジーを接続して**、キャッシュを再利用したトリプル抽出だけを走らせます。
- 抽出は LLM が行うため、結果に揺れがあります。そこで、「期待する事実」の**再現率**(しきい値は `MIN_RECALL`)と、「2 記事に共通するノードがある」ことで、合否を決めます。
- **既定は新規作成です**(`RUN_BUILD = True`)。既存グラフを読むだけの場合は `RUN_BUILD = False` にします。再利用モードでは、新しいカーネルでもセットアップと T1 の後、T9 以降の検索・描画・来歴確認へ進めます。T2〜T8 は実行されず、既存リソースの削除、アップロード、同期、オントロジーの更新は行いません。
- 新規作成から検証する場合だけ `RUN_BUILD = True` にします。作るオブジェクトは `ft-entity` で始まる名前です。**T4 で同名のナレッジ ベースとナレッジ ソースを削除し、作り直します。**
- `Run All` は選択したモードで実行され、実行したテストだけが PASS / FAIL で記録されます。T10〜T12 は、このノートブックの 2 記事とオントロジーで作成済みのグラフを対象とします。

## 生成から可視化までの実行順

1. `RUN_BUILD = True` で 1〜10 章を順に実行し、2 記事のアップロード、チャンク化、オントロジーの接続、関係抽出を行います。T4 は同名の検証用リソースを削除するため、接続先と名前を先に確認します。
2. 11〜13 章で内蔵スキーマの期待する事実・文書間の共有ノード・根拠を検証します。外部スキーマで必要な型・関係がそろわない場合、これらの固定テストは明示的にスキップします。
3. 15 章で **Foundry IQ から再取得したオントロジーと実際のエンティティ・関係を描画**し、16 章で GQL の探索・集計・根拠本文を確認します。可視化のための再取り込みは行いません。

## 既存グラフから再開する最短手順

1. 新しいカーネルで「1. セットアップ」の設定・テスト用スキーマ・検索ヘルパー・描画ヘルパーを実行します。パッケージのインストールは未導入の場合だけ必要です。
2. `RUN_BUILD = False` のまま、「2. T1: 接続」を実行します。再利用先の `KS_NAME` は既定で `ft-entity-ks` です。別のグラフを使う場合は、設定セルの名前を変更します。
3. 「10. T9: グラフ インデックス」から順に実行します。グラフ名とインデックス名は、既存のナレッジ ソースの最新定義から取得します。T10 を飛ばして T11・T12 を実行することもできます。

`Run All` でも同じ再利用経路を実行できます。描画だけなら、セットアップと T1 の次に **15 章 → 16 章** へ直接進めます。入力記事、外部 JSON-LD、ビルド成果物は不要です。既存グラフが取得できない場合はエラーで止まり、自動的に新規作成へ切り替わることはありません。

> 出典: [Foundry IQ: Introducing knowledge graphs, engineered for agents](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/foundry-iq-introducing-knowledge-graphs-engineered-for-agents/4572683) / [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md) / [azure-search-documents 12.2.0b1](https://pypi.org/project/azure-search-documents/12.2.0b1/) / [Documents - Graph(REST API 2026-11-01-preview)](https://learn.microsoft.com/en-us/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

## 作成されるリソースと関連コンポーネント

**ノートブックがナレッジ ソースを作成し、Foundry IQ がその配下の取り込み・検索コンポーネントを自動生成します。** 青はノートブックが作成・設定するもの、緑は Foundry IQ が自動生成するもの、灰色は事前に用意するものです。実線は主な処理・データの流れ、破線は設定・参照・モデル呼び出しを表します。

```mermaid
flowchart TB
    subgraph SOURCE["ソース用ストレージ (アカウントは既存)"]
        BLOB[("Blob コンテナー: ft-entity<br/>T4: 作成または再利用し、2 記事を配置")]
    end

    subgraph SEARCH["Foundry IQ (既存のサービス)"]
        KS["ナレッジ ソース: ft-entity-ks<br/>T5: graphExtraction + cache を設定して作成<br/>T8: inline オントロジーを追加"]
        subgraph AUTO["Foundry IQ が自動生成・管理: createdResources"]
            DS["データ ソース<br/>datasource: Blob の接続・読み込み定義"]
            IXR["インデクサー<br/>indexer: 取り込み処理を実行"]
            SK["スキルセット<br/>skillset: チャンク化・ベクトル化・トリプル抽出"]
            CI[("コンテンツ インデックス<br/>index: 本文チャンク + ベクトル")]
            GI[("グラフ インデックス<br/>graphIndex: トリプル + 根拠チャンク ID")]
            DS --> IXR --> SK
            SK -->|T5・T6: チャンクを格納| CI
            SK -->|T8: 抽出したエッジを格納| GI
            GI -. 根拠チャンク ID で参照 .-> CI
        end
        KB["ナレッジ ベース: ft-entity-kb (任意)<br/>T13: RUN_KB_TEST = True のとき別途作成"]
        KS -. 定義に従って自動生成・構成 .-> AUTO
        KB -. ナレッジ ソースを参照 .-> KS
    end

    subgraph AOAI["Azure OpenAI (既存のモデル デプロイ)"]
        EMB["埋め込みモデル<br/>ベクトル化"]
        CHAT["チャット モデル<br/>トリプル抽出 / GQL・回答生成"]
    end
    subgraph CACHE["キャッシュ用ストレージ (アカウントは既存)"]
        EC[("Foundry IQ が自動生成するキャッシュ<br/>Blob コンテナー + テーブル")]
    end

    BLOB -->|記事を読み込み| DS
    IXR <-->|処理結果の保存・再利用| EC
    SK -. ベクトル化を依頼 .-> EMB
    SK -. T8: トリプル抽出を依頼 .-> CHAT
    KB -. GQL・回答生成を依頼 .-> CHAT

    classDef notebook fill:#dbeafe,stroke:#2563eb,color:#172554
    classDef generated fill:#dcfce7,stroke:#16a34a,color:#14532d
    classDef existing fill:#f1f5f9,stroke:#64748b,color:#0f172a
    class BLOB,KS,KB notebook
    class DS,IXR,SK,CI,GI,EC generated
    class EMB,CHAT existing
    style KB stroke-dasharray:5 5
```

- **生成名の確認:** 図中の `datasource`、`indexer`、`skillset`、`index`、`graphIndex` は、`azureBlobParameters.createdResources` のキーです。実際のリソース名はサービスの応答から取得します。`graphIndex` は作成直後の応答に含まれない場合があるため、T8 の同期完了後に定義を再取得します。
- **オントロジーとキャッシュ:** オントロジーはナレッジ ソースの `graphExtraction.ontology` に埋め込む JSON-LD で、独立した Foundry IQ リソースではありません。T5・T6 ではチャンク化とベクトル化を行い、T8 ではキャッシュを再利用してトリプルを抽出します。グラフ名 `ft-entity` は `graphExtraction.graphName` の値で、グラフ インデックスのリソース名とは区別します。
- **自動作成されないもの:** Foundry IQ サービス、ストレージ アカウント、モデル デプロイは既存のものを使います。ナレッジ ベースもナレッジ ソースの作成に伴って自動生成されるのではなく、T13 を有効にした場合のみノートブックが別途作成します。

> 出典: [Blob ナレッジ ソースが自動生成するオブジェクト](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-blob) / [Entity Graph の生成リソースとナレッジ ベース](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#create-the-graph-enabled-knowledge-source) / [エンリッチメント キャッシュのコンテナーとテーブル](https://learn.microsoft.com/azure/search/enrichment-cache-how-to-configure) / [オントロジーの後付けとキャッシュの再利用](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md)

## テストの流れ

```mermaid
flowchart LR
    TXT["2 つのテキスト"] --> BLOB["Blob にアップロード"] --> KS["ナレッジ ソースを作成<br/>graphExtraction + キャッシュ<br/>オントロジーなし"]
    KS --> SYNC["同期<br/>チャンク化 + ベクトル化"] --> CI[("コンテンツ インデックス")]
    SYNC --> ATT["オントロジーを接続"] --> TRI["トリプル抽出<br/>キャッシュを再利用"] --> GI[("グラフ インデックス")]
    GI --> Q["GQL / 来歴 / 期待する事実"]
    CI --> Q
    GI --> VIS["登録済みオントロジー・実体を可視化"] --> EXPLORE["GQL・集計・根拠本文"]
```

## テスト一覧

| ID | 対象 | 合格条件 |
| --- | --- | --- |
| T1 | 接続 | SDK が 12.2.0b1。Foundry IQ に接続できる。`SearchClient.graph` がある |
| T2 | 入力 | 2 ファイルがあり、整形後にタグや `\n` の文字列が残らない |
| T3 | ストレージ | 入力ソースの階層型名前空間が無効、Blob の論理削除が有効、バージョン管理が無効。キャッシュ保存先は汎用 v2 |
| T4 | アップロード | 2 つの Blob が置かれる |
| T5 | ナレッジ ソースの作成(オントロジーなし) | グラフ名が設定され、オントロジーが空で、コンテンツ インデックスとインデクサーが自動生成される |
| T6 | 最初の同期 | 失敗が 0 件。チャンクができる。**グラフにエッジがまだない** |
| T7 | オントロジー | 宣言数が 128 以下、UTF-16 コード単位が 65,536 以下 |
| T8 | オントロジーの接続 | トリプルが書かれる。チャンク数が変わらない(キャッシュの再利用)。定義にオントロジーが残る |
| T9 | グラフ インデックス | `edgeList` のプロファイル。エッジがあり、すべてのエッジに根拠のチャンクが付く |
| T10 | 期待する事実 | 再現率が `MIN_RECALL` 以上 |
| T11 | 文書をまたぐ共有ノード | 所在地、出来事、指定のそれぞれで、2 つの寺社が同じノードにつながる |
| T12 | マルチホップと来歴 | 3 ホップの結果に創建者が含まれ、根拠のチャンクの本文を取得できる |
| T13 | ナレッジ ベース(任意) | 回答に、参照とグラフ検索の実行ログが付く |
| T14 | 登録内容の可視化 | 登録済みオントロジー・実体間の関係・根拠チャンクを取得できる |

## 前提

### 必要なリソース

| リソース | 用途 |
| --- | --- |
| Foundry IQ(ナレッジ ソースとナレッジ ベースを作成できるリージョン) | ナレッジ ソース、インデックス |
| Azure OpenAI(チャット モデルと埋め込みモデルのデプロイ) | グラフ抽出、ベクトル化 |
| Blob Storage(ソース用、階層型名前空間は無効) | 記事テキストの配置。最初の取り込み前に Blob の論理削除を有効化し、バージョン管理を無効化します |
| 汎用 v2 ストレージ アカウント(キャッシュ用、`StorageV2`) | エンリッチメント キャッシュの Blob コンテナーとテーブル |

Entity Graph の公式手順は、**入力ソースとして ADLS Gen2 をサポートしません**。また、最初の取り込み前にネイティブな Blob の論理削除を有効化し、Blob のバージョン管理を無効化することを要求しています。T3 でこれらの設定を確認し、不適合なら削除・アップロード・ナレッジ ソース作成を止めます。

キャッシュ保存先の公式要件は汎用 v2 です。キャッシュ保存先の階層型名前空間を無効化するという条件は、引用した公式資料では確認できないため、入力ソースの制限と混同しません。実行時にサービスが返すエラーは、そのまま確認してください。

> 出典: [Entity Graph の前提条件](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#prerequisites) / [エンリッチメント キャッシュの前提条件](https://learn.microsoft.com/azure/search/enrichment-cache-how-to-configure#prerequisites)

### ロール

| 付与先 | ロール | スコープ |
| --- | --- | --- |
| 実行するユーザー | Search Service Contributor、Search Index Data Contributor | Foundry IQ サービス |
| 実行するユーザー | Storage Blob Data Contributor | ソース用ストレージ |
| Foundry IQ のマネージド ID | Storage Blob Data Reader | ソース用ストレージ |
| Foundry IQ のマネージド ID | Storage Blob Data Contributor、Storage Table Data Contributor | キャッシュ用ストレージ |
| Foundry IQ のマネージド ID | Cognitive Services OpenAI User | Azure OpenAI |

### `.env`

- **既存グラフの再利用:** `AZURE_SEARCH_ENDPOINT` のみ必須です。入力ファイル、ストレージの管理権限、埋め込みモデルの設定は不要です。上の作成用リソースとロールの一覧は、新規作成時の前提です。再利用には、ナレッジ ソース・インデックスの定義の読み取りと、インデックスの検索ができる権限が必要です。
- **新規作成:** 追加で `AZURE_OPENAI_ENDPOINT`、`AZURE_OPENAI_DEPLOYMENT`、`AZURE_OPENAI_EMBEDDING_DEPLOYMENT`、`AZURE_STORAGE_RESOURCE_ID`、`AZURE_CACHE_STORAGE_RESOURCE_ID` が必要です(任意で `AZURE_OPENAI_MODEL`、`AZURE_OPENAI_EMBEDDING_MODEL`)。
- **T13 の実行:** 再利用モードでも、`AZURE_OPENAI_ENDPOINT` と `AZURE_OPENAI_DEPLOYMENT` が必要です。ナレッジ ベースを作成・更新するための権限も必要です。

> 出典: [Create a Blob Knowledge Source for Agentic Retrieval](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-blob) / [Connect existing resources(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-provision/references/connect-existing-resources.md)

## 1. セットアップ

### 1.1 パッケージのインストール

ナレッジ グラフのモデルと API(`GraphProfile`、`KnowledgeSourceGraphExtractionParameters`、`SearchClient.graph` など)は、`azure-search-documents` 12.2.0b1 で追加されています。バージョンを固定しているので、`--pre` は付けません。

> 出典: [azure-search-documents CHANGELOG](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/CHANGELOG.md)

In [ ]:
%pip install -q azure-search-documents==12.2.0b1 azure-identity azure-storage-blob python-dotenv pandas networkx matplotlib requests

### 1.2 設定

`.env` から接続先を読み込みます。認証は Microsoft Entra ID(`DefaultAzureCredential`)です。`check()` は、テストの結果を `results` に記録します。原則として失敗も最後にまとめて表示しますが、T3 のストレージ要件を満たさない場合は例外で停止します。T4・T5 は、現在の入力ソースとキャッシュ保存先について T3 が成功していなければ書き込みません。

| 名前 | 意味 |
| --- | --- |
| `RUN_BUILD` | 既定は `True` です。T2〜T8 を実行し、T4 で同名リソースを削除して再作成します。既存グラフを検索するだけなら `False` にします |
| `KS_NAME` | 既定は `ft-entity-ks` です。別の既存グラフを使う場合は変更します |
| `ONTOLOGY_PATH` | 既定の `None` は内蔵オントロジーです。外部の検証済み JSON-LD を使う場合だけパスを指定します |
| `MAX_DISPLAY_EDGES` | 登録済みグラフの描画に取得する文書の上限です |
| `MIN_RECALL` | 「期待する事実」のうち、グラフに見つかった割合の下限。LLM の抽出には揺れがあるため、100% は求めません |
| `RUN_KB_TEST` | `True` にすると、ナレッジ ベースの機能テスト(T13)も実行します。モデルの費用が追加で発生します |
| `RUN_CLEANUP` | `True` にすると、新規作成モードで作成したオブジェクトを最後に削除します。再利用モードでは削除できません |

In [ ]:
import hashlib
import json
import logging
import os
import re
import time
from importlib.metadata import version
from pathlib import Path

from IPython.display import display

import pandas as pd
import requests
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(override=True)
credential = DefaultAzureCredential()

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
API_VERSION = "2026-11-01-preview"
RUN_BUILD = False
RUN_KB_TEST = False
RUN_CLEANUP = False
VALIDATED_STORAGE_IDS = None

FT = "ft-entity"  # このテストが作るオブジェクトの名前の接頭辞
GRAPH_NAME = FT
KS_NAME = f"{FT}-ks"
KB_NAME = f"{FT}-kb"
CONTAINER = FT
MIN_RECALL = 0.6
ONTOLOGY_PATH = None  # 外部 JSON-LD を使う場合だけ Path(...) を指定します
MAX_DISPLAY_EDGES = 200
if not 1 <= MAX_DISPLAY_EDGES <= 1000:
    raise ValueError("MAX_DISPLAY_EDGES は 1 から 1000 の範囲にしてください。")

if RUN_BUILD or RUN_KB_TEST:
    AOAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
    CHAT_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
    CHAT_MODEL = os.environ.get("AZURE_OPENAI_MODEL", CHAT_DEPLOYMENT)
if RUN_BUILD:
    EMBED_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
    EMBED_MODEL = os.environ.get("AZURE_OPENAI_EMBEDDING_MODEL", EMBED_DEPLOYMENT)
    SOURCE_STORAGE_ID = os.environ["AZURE_STORAGE_RESOURCE_ID"]
    CACHE_STORAGE_ID = os.environ["AZURE_CACHE_STORAGE_RESOURCE_ID"]

results = []


def check(case, ok, detail=""):
    results.append({"テスト": case, "結果": "PASS" if ok else "FAIL", "詳細": str(detail)[:200]})
    print("PASS" if ok else "FAIL", "-", case, f"| {detail}" if detail else "")
    return ok


print("実行モード:", "新規作成(T4 で同名リソースを削除)" if RUN_BUILD else "既存グラフの再利用(T2〜T8 はスキップ)")
pd.DataFrame({"値": [SEARCH_ENDPOINT, KS_NAME]}, index=["Foundry IQ", "ナレッジ ソース"])

### 1.3 テスト用スキーマ

新規作成時の抽出と、T10 の関係別検索が共通して使うクラス・関係・属性です。再利用時もこのセルを実行します。ここではローカルな表を定義するだけで、既存のオントロジーやグラフは変更しません。JSON-LD への組み立てと接続は、新規作成モードの T7・T8 で行います。

> 出典: [Entity Graph のオントロジー](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph)

In [ ]:
classes = pd.DataFrame([
    ("ReligiousSite", "神社や寺院などの施設。例: 花ケ咲神社、近藤寺"),
    ("Deity", "神社の祭神や寺院の本尊として祀られる神仏。例: 桜花女神、百八面千手観世音菩薩"),
    ("Person", "人物。例: 近藤智海、織田信長"),
    ("Facility", "境内にある建造物、庭園、茶室。例: 本殿、金堂、茶室「阿須杯庵」、石庭「舞供楼の庭」"),
    ("Place", "地名や行政区画。例: 京都市東山区"),
    ("Event", "祭事、行事、歴史上の出来事。例: 桜花祭、応仁の乱"),
    ("Designation", "世界遺産などの指定や登録。例: 古都京都の文化財"),
], columns=["class", "comment"])

relations = pd.DataFrame([
    ("ReligiousSite", "locatedIn", "Place", "施設が所在する地域"),
    ("ReligiousSite", "enshrines", "Deity", "施設が祀る祭神や本尊"),
    ("ReligiousSite", "foundedBy", "Person", "施設を創建した人物"),
    ("ReligiousSite", "hasFacility", "Facility", "施設の境内にある建造物や庭園"),
    ("ReligiousSite", "survivedEvent", "Event", "施設が焼失を免れた出来事"),
    ("ReligiousSite", "registeredAs", "Designation", "施設が登録された指定"),
    ("Person", "visited", "Facility", "人物が訪れた建造物や庭園"),
], columns=["domain", "predicate", "range", "comment"])

datatype_properties = pd.DataFrame([("ReligiousSite", "foundedYear", "integer", "施設が創建された西暦年")], columns=["domain", "predicate", "xsd_range", "comment"])
relations

### 1.4 検索ヘルパー

T10〜T12 で使う GQL 実行・結果の表形式への変換・名前の表記ゆれの正規化を定義します。来歴を取得する SDK モデルもここで読み込みます。検索先とグラフ名のフィルターは T9 で設定します。定義だけでは検索は実行しません。

> 出典: [Documents - Graph](https://learn.microsoft.com/en-us/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
from azure.search.documents.models import GraphQueryProvenanceHydrateDocumentsParameters, GraphQueryProvenanceParameters


def run_gql(gql, provenance=None):
    result = graph_client.graph(gql=gql, filter=GRAPH_FILTER, include_statistics=True, provenance=provenance)
    print(gql, "\n   行数:", result.count)
    return result


def rows(result):
    return pd.DataFrame([r.as_dict() for r in result.value]).drop(columns=["@search.graphProvenance"], errors="ignore")


def norm(s):
    return re.sub(r"[\s·・･]", "", s or "")

### 1.5 描画ヘルパー

オントロジー、検索したグラフ全体、マルチホップの経路で共通して使う描画処理です。新規作成の有無によらず、このセルを実行します。

同じノード間の複数の関係も描画できるよう、公開内容の図には `MultiDiGraph` を使います。

> 出典: [NetworkX のグラフ描画](https://networkx.org/documentation/stable/reference/drawing.html)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import Patch

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
matplotlib.rcParams["font.sans-serif"] = ["Yu Gothic", "Meiryo", "MS Gothic", "Noto Sans CJK JP", "Noto Sans JP", "Hiragino Sans", "DejaVu Sans"]


import textwrap

def draw_graph(graph, title, node_size=1400, figsize=(13, 9), positions=None):
    if not graph:
        raise ValueError("描画対象の関係がありません。取得件数とオントロジーを確認してください。")
    kinds = sorted({data.get("kind", data.get("type", "?")) for _, data in graph.nodes(data=True)})
    palette = {kind: plt.cm.tab20(number % 20) for number, kind in enumerate(kinds)}
    if positions is None:
        positions = nx.spring_layout(graph, seed=7, k=1.5)
    styles = [f"arc3,rad={number * 0.12}" for number in range(10)]
    fig, ax = plt.subplots(figsize=figsize)
    nx.draw_networkx_nodes(graph, positions, ax=ax, node_size=node_size, alpha=0.9,
                           node_color=[palette[graph.nodes[node].get("kind", graph.nodes[node].get("type", "?"))] for node in graph])
    nx.draw_networkx_edges(graph, positions, ax=ax, arrows=True, arrowsize=15, node_size=node_size, connectionstyle=styles, edge_color="#777777")
    nx.draw_networkx_labels(graph, positions, ax=ax, font_size=9,
                            labels={node: textwrap.fill(str(data.get("label", node)), 15) for node, data in graph.nodes(data=True)},
                            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1.5})
    nx.draw_networkx_edge_labels(graph, positions, ax=ax, edge_labels=nx.get_edge_attributes(graph, "predicate"),
                                 font_size=7, rotate=False, node_size=node_size, connectionstyle=styles)
    ax.legend(handles=[Patch(color=palette[kind], label=kind) for kind in kinds], loc="upper left")
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

### 1.6 外部 JSON-LD の検証（任意）

`ONTOLOGY_PATH = None` なら次の検証・変換セルは実行されず、ランタイムも不要です。外部ファイルを指定する場合は、先に [README の手動取得・配置手順](./README.md) に従って公式の `foundry-iq-skills` を取得します。この配布セットには同梱しておらず、自動ダウンロードも行いません。別の場所に配置した場合は `FOUNDRY_IQ_PLUGIN_ROOT` で指定してください。入力ファイルは変更せず、`RUN_BUILD = True` の作成経路で検証・変換した内容を使用します。

> 出典: [オントロジーの変換処理](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels)

外部ファイルの不正を、リソースの削除・作成やモデル呼び出しより前に検出するため、指定した場合はセットアップで先に検証します。

In [ ]:
if RUN_BUILD and ONTOLOGY_PATH is not None:
    import subprocess
    import sys

    ONTOLOGY_PATH = Path(ONTOLOGY_PATH).expanduser().resolve()
    plugin_root = Path(os.environ.get("FOUNDRY_IQ_PLUGIN_ROOT", "foundry-iq-skills")).expanduser().resolve()
    BOOTSTRAP = plugin_root / "graph" / "runtime" / "bootstrap.py"
    if not BOOTSTRAP.is_file():
        raise FileNotFoundError(f"Foundry IQ ランタイムが未配置です: {BOOTSTRAP}。README の手動取得・配置手順に従い、foundry-iq-skills をノートブックと同じ階層に置くか、FOUNDRY_IQ_PLUGIN_ROOT に配置先を指定してください。自動ダウンロードは行いません。")
    print("検証ランタイム:", BOOTSTRAP)

In [ ]:
if RUN_BUILD and ONTOLOGY_PATH is not None:
    if not ONTOLOGY_PATH.is_file():
        raise FileNotFoundError(f"{ONTOLOGY_PATH} がありません。ONTOLOGY_PATH を確認してください。内蔵オントロジーを使う場合は None にしてください。")
    if not BOOTSTRAP.is_file():
        raise FileNotFoundError(f"ランタイムがありません: {BOOTSTRAP.resolve()}。設定セルを再実行し、表示される検証ランタイムを確認してください。")
    validation = subprocess.run([sys.executable, "-I", str(BOOTSTRAP.resolve()), "run", "ontology", "ontology-consume", "check", str(ONTOLOGY_PATH.resolve()), "--json"],
                                capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=120)
    print(validation.stdout, validation.stderr)
    validation.check_returncode()
    profile = json.loads(validation.stdout)
    if profile["status"] != "ok" or profile["violations"]:
        raise ValueError("オントロジーのプロファイル検証が通っていません。")
    ontology_text = ONTOLOGY_PATH.read_text(encoding="utf-8")
    ontology_hash = hashlib.sha256(ontology_text.encode("utf-8")).hexdigest()
    print("オントロジー SHA-256:", ontology_hash)

### 1.7 外部 JSON-LD を Foundry IQ に送る JSON-LD を作る

生成ファイルの `urn:ontology:...#Temple` のような識別子は、JSON-LD として有効でも Foundry IQ に拒否される場合があります。手動取得した Entity Graph ランタイムの `load_ontology(INLINE_JSONLD)` を使い、埋め込みコンテキストで定義された `entity:Temple` のような識別子へ変換します。登録用の名前空間は `https://schema.search.azure.com/graph-extraction#` になります。

クラス・関係・属性、domain / range、抽出用の説明を引き継ぎ、オントロジー ヘッダーなどの管理メタデータは送信用定義に含めません。**元ファイルや承認済みの版は変更しません。** 入力ファイルの `ontology_hash` と、実際に送る `search_ontology_hash` は別に記録します。登録後は後者と照合します。

変換コードは、取得したランタイムのマニフェストと SHA-256 を照合した wheel から、隔離した Python プロセスで呼びます。子プロセスの UTF-8 モードも明示し、日本語の説明を保持します。この処理は標準ライブラリのみを使い、追加インストールや Azure・モデルへの呼び出しを行いません。

> 出典: [Foundry IQ Skills の公式配布コード（`ekg_core.ontology.load_ontology` / `_search_inline`）](https://github.com/microsoft/azure-skills/tree/main/.github/plugins/foundry-iq-skills/graph/runtime/wheels) / [Entity Graph の埋め込みコンテキストの例](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#create-the-graph-enabled-knowledge-source)

In [ ]:
if RUN_BUILD and ONTOLOGY_PATH is not None:
    runtime_dir = BOOTSTRAP.resolve().parent
    runtime_manifest = json.loads((runtime_dir / "manifest.json").read_text(encoding="utf-8"))
    adapter_record = runtime_manifest["runtimes"]["entity"]["wheel"]
    adapter_wheel = (runtime_dir / adapter_record["file"]).resolve()
    if not adapter_wheel.is_relative_to(runtime_dir):
        raise ValueError("変換用 wheel がランタイムのフォルダー外を指しています。")
    if hashlib.sha256(adapter_wheel.read_bytes()).hexdigest() != adapter_record["sha256"]:
        raise ValueError("変換用 wheel の SHA-256 がマニフェストと一致しません。")

    import textwrap
    adapter_script = textwrap.dedent("""
    import json, sys
    sys.path.insert(0, sys.argv[1])
    from ekg_core.models import OntologyMode
    from ekg_core.ontology import load_ontology
    ontology = load_ontology(OntologyMode.INLINE_JSONLD, path=sys.argv[2])
    print(json.dumps({"source_sha256": ontology.schema_id, "inline_jsonld": ontology.inline_jsonld}, ensure_ascii=False))
    """)
    conversion = subprocess.run([sys.executable, "-I", "-X", "utf8", "-c", adapter_script, str(adapter_wheel), str(ONTOLOGY_PATH.resolve())],
                                capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=120)
    if conversion.returncode:
        print(conversion.stdout, conversion.stderr)
    conversion.check_returncode()
    converted = json.loads(conversion.stdout)
    if converted["source_sha256"] != f"sha256:{ontology_hash}":
        raise RuntimeError("検証後に入力ファイルが変わっています。元ファイルの検証セルから再実行してください。")
    search_ontology_text = converted["inline_jsonld"]
    search_ontology_hash = hashlib.sha256(search_ontology_text.encode("utf-8")).hexdigest()
    display(pd.DataFrame([{"形式": "元ファイル（変更なし）", "SHA-256": ontology_hash},
                          {"形式": "Foundry IQ に送る JSON-LD", "SHA-256": search_ontology_hash}]))
    print("Foundry IQ 送信用コンテキスト:", json.loads(search_ontology_text)["@context"])

## 2. T1: 接続

SDK のバージョンと、Foundry IQ への接続、グラフ クエリの API を確かめます。

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient

index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, api_version=API_VERSION)
indexer_client = SearchIndexerClient(SEARCH_ENDPOINT, credential, api_version=API_VERSION)

existing = [ks.name for ks in index_client.list_knowledge_sources()]
check("T1-1 SDK が 12.2.0b1", version("azure-search-documents") == "12.2.0b1", version("azure-search-documents"))
check("T1-2 Foundry IQ に接続し、ナレッジ ソースを一覧できる", True, f"既存のナレッジ ソース {len(existing)} 件")
check("T1-3 グラフ クエリ(SearchClient.graph)を使える", hasattr(SearchClient, "graph"))

## 3. T2: 入力のテキスト

`近藤寺 - Hanapedia.txt` と `花ケ咲神社 - Hanapedia.txt` は、架空の百科事典「ハナペディア」の記事です。2 記事には、次の共通点があり、文書をまたぐ質問に使えます。

| 共通点 | 近藤寺 | 花ケ咲神社 |
| --- | --- | --- |
| 所在地 | 京都市東山区 | 京都市東山区 |
| 世界遺産 | 古都京都の文化財 | 古都京都の文化財 |
| 歴史上の出来事 | 応仁の乱の焼失を免れた | 応仁の乱の焼失を免れた |

ファイルの拡張子は `.txt` ですが、内容は見出し・段落・表を使った Markdown です。改行は実際の改行として保存されており、ページ情報の HTML コメントやエスケープされた改行はありません。既存の整形処理を通してからアップロードします。Blob 名は ASCII にします。

In [ ]:
if RUN_BUILD:
    DATA_DIR = Path(".")  # ノートブックと同じフォルダー
    SAMPLES = {"kondoji.txt": "近藤寺 - Hanapedia.txt", "hanagasaki-jinja.txt": "花ケ咲神社 - Hanapedia.txt"}


def clean_text(raw):
    text = raw.replace("\\n", "\n").replace('\\"', '"')
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    text = re.sub(r"<tr>(.*?)</tr>", lambda m: " / ".join(re.findall(r"<td>(.*?)</td>", m.group(1), flags=re.S)) + "\n", text, flags=re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"(?<=[\u3000-\u9fff、。」』)])\n(?=[\u3000-\u9fffA-Za-z「『])", "", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


if RUN_BUILD:
    raw_texts = {blob: (DATA_DIR / src).read_text(encoding="utf-8") for blob, src in SAMPLES.items()}
    cleaned = {blob: clean_text(raw) for blob, raw in raw_texts.items()}

    check("T2-1 2 つの入力ファイルがある", all((DATA_DIR / src).is_file() for src in SAMPLES.values()), list(SAMPLES.values()))
    check("T2-2 整形後に、タグと `\\n` の文字列が残らない", all("<" not in t and "\\n" not in t for t in cleaned.values()))
    check("T2-3 整形後の文章が 1,000 文字以上ある", all(len(t) > 1000 for t in cleaned.values()), {b: len(t) for b, t in cleaned.items()})
    display(pd.DataFrame([{"Blob 名": b, "元ファイル": SAMPLES[b], "整形前": len(raw_texts[b]), "整形後": len(cleaned[b])} for b in SAMPLES]))

## 4. T3: ストレージの状態

入力ソースとキャッシュ保存先の設定を ARM から読み取り、用途ごとに異なる要件を検証します。設定の変更は自動で行いません。

| 用途 | 必須条件 | 根拠 |
| --- | --- | --- |
| 入力ソース | 階層型名前空間が無効、ネイティブな Blob の論理削除が有効、Blob のバージョン管理が無効 | Entity Graph の前提条件 |
| キャッシュ保存先 | 汎用 v2 ストレージ アカウント(`StorageV2`) | エンリッチメント キャッシュの前提条件 |

キャッシュ保存先の階層型名前空間・論理削除・バージョン管理は表示だけに使い、入力ソースの条件を適用しません。ARM の `isHnsEnabled` と `isVersioningEnabled` は `true` のときに有効となる設定です。いずれかの必須条件が不適合なら、結果を表示したうえで停止します。設定を修正した場合やストレージのリソース ID を変更した場合は、T3 を再実行してください。

ネットワーク設定は参考情報です。ARM の設定を読み取れたことだけでは、Foundry IQ からの到達性を確認したことにはなりません。HTTP エラーは設定の読み取り前に例外として返します。

> 出典: [Entity Graph の前提条件](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#prerequisites) / [エンリッチメント キャッシュの前提条件](https://learn.microsoft.com/azure/search/enrichment-cache-how-to-configure#prerequisites) / [ストレージ アカウントの ARM 設定](https://learn.microsoft.com/rest/api/storagerp/storage-accounts/get-properties?view=rest-storagerp-2024-01-01) / [Blob サービスの ARM 設定](https://learn.microsoft.com/rest/api/storagerp/blob-services/get-service-properties?view=rest-storagerp-2024-01-01)

In [ ]:
if RUN_BUILD:
    VALIDATED_STORAGE_IDS = None
    arm = {"Authorization": f"Bearer {credential.get_token('https://management.azure.com/.default').token}"}
    storage_rows = []
    for label, rid in [("ソース", SOURCE_STORAGE_ID), ("キャッシュ", CACHE_STORAGE_ID)]:
        account_response = requests.get(f"https://management.azure.com{rid}?api-version=2024-01-01", headers=arm, timeout=30)
        account_response.raise_for_status()
        account_data = account_response.json()
        account = account_data["properties"]
        blob_response = requests.get(f"https://management.azure.com{rid}/blobServices/default?api-version=2024-01-01", headers=arm, timeout=30)
        blob_response.raise_for_status()
        blob = blob_response.json()["properties"]
        storage_rows.append({"用途": label, "アカウント": rid.split("/")[-1], "アカウントの種類": account_data.get("kind"),
                             "階層型名前空間": bool(account.get("isHnsEnabled")),
                             "publicNetworkAccess": account.get("publicNetworkAccess"), "defaultAction": (account.get("networkAcls") or {}).get("defaultAction"),
                             "Blob の論理削除": bool((blob.get("deleteRetentionPolicy") or {}).get("enabled")),
                             "バージョン管理": bool(blob.get("isVersioningEnabled"))})
    storage = pd.DataFrame(storage_rows)
    display(storage)
    source_storage = storage.loc[storage["用途"] == "ソース"].iloc[0]
    cache_storage = storage.loc[storage["用途"] == "キャッシュ"].iloc[0]
    storage_checks = [
        check("T3-1 入力ソースの階層型名前空間が無効", not source_storage["階層型名前空間"], source_storage["階層型名前空間"]),
        check("T3-2 入力ソースの Blob の論理削除が有効", bool(source_storage["Blob の論理削除"]), source_storage["Blob の論理削除"]),
        check("T3-3 入力ソースの Blob のバージョン管理が無効", not source_storage["バージョン管理"], source_storage["バージョン管理"]),
        check("T3-4 キャッシュ保存先が汎用 v2", cache_storage["アカウントの種類"] == "StorageV2", cache_storage["アカウントの種類"]),
    ]
    if not all(storage_checks):
        raise RuntimeError("T3 のストレージ要件を満たしていません。FAIL の設定を修正し、T3 を再実行してください。削除・アップロード・ナレッジ ソース作成は実行しません。")
    VALIDATED_STORAGE_IDS = (SOURCE_STORAGE_ID, CACHE_STORAGE_ID)

## 5. T4: 前回の掃除と、Blob へのアップロード

前回の実行が残っていれば、`ft-entity` で始まるナレッジ ベースとナレッジ ソースを削除します。ナレッジ ソースを削除すると、自動生成されたデータ ソース、スキルセット、インデクサー、インデックスも一緒に削除されます。削除の反映を待ってから、整形したテキストを Blob コンテナーへ置きます。

In [ ]:
from azure.storage.blob import BlobServiceClient, ContentSettings

if RUN_BUILD:
    if VALIDATED_STORAGE_IDS != (SOURCE_STORAGE_ID, CACHE_STORAGE_ID):
        raise RuntimeError("現在のストレージについて T3 の検証を先に成功させてください。T4 の削除・アップロードは実行しません。")
    for delete, name in ((index_client.delete_knowledge_base, KB_NAME), (index_client.delete_knowledge_source, KS_NAME)):
        try:
            delete(name)
        except ResourceNotFoundError:
            pass
    for _ in range(60):  # 削除の反映を待ちます
        try:
            index_client.get_knowledge_source(KS_NAME)
            time.sleep(5)
        except ResourceNotFoundError:
            break

    blob_service = BlobServiceClient(f"https://{SOURCE_STORAGE_ID.split('/')[-1]}.blob.core.windows.net", credential)
    container = blob_service.get_container_client(CONTAINER)
    if not container.exists():
        container.create_container()
    for blob_name, text in cleaned.items():
        container.upload_blob(blob_name, text.encode("utf-8"), overwrite=True, content_settings=ContentSettings(content_type="text/plain; charset=utf-8"))

    blobs = [b.name for b in container.list_blobs()]
    check("T4 2 つの Blob が置かれる", set(blobs) == set(SAMPLES), blobs)

## 6. T5: ナレッジ ソースの作成(オントロジーなし)

Azure Blob のナレッジ ソースを作ると、Foundry IQ が次のオブジェクトを自動生成します(データ ソース、スキルセット、コンテンツ インデックス、インデクサー)。次の指定を加えます。

| 指定 | 意味 |
| --- | --- |
| `is_adls_gen2=False` | Entity Graph の入力を標準の Azure Blob Storage として指定します。実際の階層型名前空間の設定は T3 で検証します |
| `graph_extraction=KnowledgeSourceGraphExtractionParameters(graph_name=...)` | グラフ抽出を有効にし、グラフに名前を付けます。**この時点では、オントロジーを付けません** |
| `cache=KnowledgeSourceCache(..., enable_reprocessing=True)` | エンリッチメント キャッシュを有効にします。オントロジーの接続時に、チャンク化とベクトル化をやり直さず、キャッシュから再処理します |
| `content_extraction_mode="minimal"`、`disable_image_verbalization=True` | テキストだけを扱います |

作成を呼ぶと、最初の同期がすぐに始まります。

> 出典: [Create a Blob Knowledge Source for Agentic Retrieval](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-blob) / [SearchServiceCreateKnowledgeSourceAzureBlobGraphCache.json(REST 仕様の例)](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceCreateKnowledgeSourceAzureBlobGraphCache.json) / [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md)

### 登録先の容量確認

コンテンツとグラフ用にインデックス 2 個、ナレッジ ソース・データ ソース・インデクサー・スキルセットを各 1 個見込みます。容量の確認と実際の作成は別の操作です。

> 出典: [サービス統計](https://learn.microsoft.com/rest/api/searchservice/get-service-statistics/get-service-statistics?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if RUN_BUILD:
    counters = index_client.get_service_statistics().counters
    requirements = [("インデックス", counters.index_counter, 2),
                    ("ナレッジ ソース", counters.knowledge_source_counter, 1),
                    ("データ ソース", counters.data_source_counter, 1),
                    ("インデクサー", counters.indexer_counter, 1),
                    ("スキルセット", counters.skillset_counter, 1)]
    if any(counter is None or counter.quota is None for _, counter, _ in requirements):
        raise RuntimeError("必要な容量上限を確認できません。書き込み前に管理者へ確認してください。")
    capacity = pd.DataFrame([{"対象": name, "使用中": counter.usage, "上限": counter.quota, "今回必要": needed}
                             for name, counter, needed in requirements])
    display(capacity)
    if ((capacity["上限"] - capacity["使用中"]) < capacity["今回必要"]).any():
        raise RuntimeError("新しい登録に必要な容量が足りません。既存リソースは自動削除しません。")

In [ ]:
from azure.search.documents.indexes.models import (
    AzureBlobKnowledgeSource,
    AzureBlobKnowledgeSourceParameters,
    AzureOpenAIVectorizerParameters,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceGraphExtractionParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceCache,
    KnowledgeSourceIngestionParameters,
)

if RUN_BUILD:
    if VALIDATED_STORAGE_IDS != (SOURCE_STORAGE_ID, CACHE_STORAGE_ID):
        raise RuntimeError("現在のストレージについて T3 の検証を先に成功させてください。ナレッジ ソース作成は実行しません。")
    embedding_params = AzureOpenAIVectorizerParameters(resource_url=AOAI_ENDPOINT, deployment_name=EMBED_DEPLOYMENT, model_name=EMBED_MODEL)
    chat_params = AzureOpenAIVectorizerParameters(resource_url=AOAI_ENDPOINT, deployment_name=CHAT_DEPLOYMENT, model_name=CHAT_MODEL)

    knowledge_source = AzureBlobKnowledgeSource(
        name=KS_NAME,
        description="京都の寺社を紹介するハナペディアの記事(近藤寺、花ケ咲神社)。歴史、祭神、境内の施設、文化財、関連する人物を含む。",
        azure_blob_parameters=AzureBlobKnowledgeSourceParameters(
            connection_string=f"ResourceId={SOURCE_STORAGE_ID}",
            container_name=CONTAINER,
            is_adls_gen2=False,
            ingestion_parameters=KnowledgeSourceIngestionParameters(
                embedding_model=KnowledgeSourceAzureOpenAIVectorizer(azure_open_ai_parameters=embedding_params),
                chat_completion_model=KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=chat_params),
                disable_image_verbalization=True,
                content_extraction_mode="minimal",
                cache=KnowledgeSourceCache(storage_connection_string=f"ResourceId={CACHE_STORAGE_ID}", enable_reprocessing=True),
            ),
            graph_extraction=KnowledgeSourceGraphExtractionParameters(graph_name=GRAPH_NAME),
        ),
    )

    created = index_client.create_or_update_knowledge_source(knowledge_source)
    resources = created.azure_blob_parameters.created_resources
    graph_extraction = created.azure_blob_parameters.graph_extraction

    check("T5-1 グラフ名が設定され、オントロジーが空である", graph_extraction.graph_name == GRAPH_NAME and graph_extraction.ontology is None, f"graph_name={graph_extraction.graph_name} ontology={graph_extraction.ontology}")
    check("T5-2 コンテンツ インデックスとインデクサーが自動生成される", bool(resources.get("index")) and bool(resources.get("indexer")), {k: v for k, v in resources.items() if v})
    display(pd.DataFrame({"自動生成されたオブジェクト": {k: v for k, v in resources.items() if v}}).rename_axis("種類"))

## 7. T6: 最初の同期(チャンク化とベクトル化だけ)

`get_knowledge_source_status` は、実行中の同期(`current_synchronization_state`)と、直近に完了した同期(`last_synchronization_state`)を返します。実行中が `None` で、直近の完了が存在すれば、同期は終わっています。

ナレッジ ソースの状態にはエラー文が含まれないため、生成されたインデクサーの最新実行も併せて表示します。`transientFailure` が続くときは、このエラー文が原因(ストレージへの到達性、ロール不足など)を示します。

> 出典: [KnowledgeSourceStatus(azure-search-documents 12.2.0b1)](https://pypi.org/project/azure-search-documents/12.2.0b1/) / [Status while a step runs(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/status-and-telemetry.md)

In [ ]:
def wait_sync(done, timeout=1800, interval=30, max_error_polls=10, rerun=False):
    started, error_polls, reruns = time.time(), 0, set()
    while time.time() - started < timeout:
        status = index_client.get_knowledge_source_status(KS_NAME)
        current, last = status.current_synchronization_state, status.last_synchronization_state
        run = indexer_client.get_indexer_status(resources["indexer"]).last_result
        shown = current or last
        print(f"{int(time.time() - started):>5}s 状態={status.synchronization_status} 実行中={'あり' if current else 'なし'} "
              f"処理済み={getattr(shown, 'items_updates_processed', '-')} 失敗={getattr(shown, 'items_updates_failed', '-')} "
              f"インデクサー={getattr(run.status, 'value', run.status) if run else '-'}")
        indexer_error = run.error_message if run else None
        if indexer_error:
            print("   インデクサーのエラー:", indexer_error[:300])
        for error in (run.errors if run else None) or []:
            print("   ドキュメントのエラー:", (error.error_message or "")[:300])
        if done(current, last):
            return status
        if rerun and indexer_error and "must be retried" in indexer_error.lower() and run.start_time not in reruns and len(reruns) < 5:
            reruns.add(run.start_time)  # 処理待ちの文書が残っているため、サービスが再実行を求めている
            try:
                indexer_client.run_indexer(resources["indexer"])
            except HttpResponseError as e:
                if e.status_code != 409:
                    raise
            print("   サービスの要求に従い、インデクサーを再実行しました")
            error_polls = 0
        else:
            error_polls = error_polls + 1 if indexer_error else 0
        if error_polls >= max_error_polls:
            raise RuntimeError("インデクサーのエラーが続いているため中断します。確認する点: "
                               "(1) キャッシュ用ストレージが階層型名前空間(ADLS Gen2)を無効にした標準の Blob か、"
                               "(2) Foundry IQ のマネージド ID にストレージのロール(キャッシュ用は Storage Table Data Contributor を含む)が付いているか、"
                               "(3) ストレージが信頼された Microsoft サービスの例外(bypass=AzureServices)を許可し、Foundry IQ から到達できるか。\n" + indexer_error[:300])
        time.sleep(interval)
    raise TimeoutError("同期が時間内に完了しませんでした")


if RUN_BUILD:
    status = wait_sync(lambda current, last: current is None and last is not None)
    last = status.last_synchronization_state
    display(pd.DataFrame([{"開始": last.start_time, "終了": last.end_time, "処理": last.items_updates_processed, "失敗": last.items_updates_failed}]))

### 7.1 コンテンツ インデックスと、グラフ インデックスの状態

生成されたコンテンツ インデックスには、チャンク(`snippet`)とそのベクトル(`snippet_vector`)が入っています。グラフのエッジが指す根拠は、このインデックスのドキュメントです。**オントロジーがない間は、トリプル抽出が走らない**ので、グラフ インデックスは、まだない(あっても、エッジが 0 件)はずです。

In [ ]:
if RUN_BUILD:
    content_index = index_client.get_index(resources["index"])
    content_client = SearchClient(SEARCH_ENDPOINT, content_index.name, credential)
    chunks_before = content_client.get_document_count()

    graph_index_before = index_client.get_knowledge_source(KS_NAME).azure_blob_parameters.created_resources.get("graphIndex")
    edges_before = SearchClient(SEARCH_ENDPOINT, graph_index_before, credential).get_document_count() if graph_index_before else 0

    check("T6-1 同期の失敗が 0 件", last.items_updates_failed == 0, f"処理 {last.items_updates_processed} / 失敗 {last.items_updates_failed}")
    check("T6-2 チャンクができる", chunks_before >= 2, f"{chunks_before} 件")
    check("T6-3 オントロジーがない間は、グラフにエッジがない", edges_before == 0, f"graphIndex={graph_index_before} / エッジ {edges_before}")
    display(pd.DataFrame(list(content_client.search("*", select=["uid", "snippet_parent_id", "blob_url", "snippet"], top=3))))

## 8. T7: オントロジーの定義

**オントロジー**は、グラフに入れるエンティティの種類(クラス)と、その間の関係(プロパティ)の設計図です。抽出用の LLM は、これに沿って、チャンクから主語・述語・目的語を取り出します。`rdfs:comment` の説明文と例が、そのまま抽出の指示になります。

形式は JSON-LD(OWL の語彙)です。サービスは、接続時に検証し、不正なら拒否します。主な制約は、次のとおりです。

- トップレベルは `@context` と `@graph` のみ。`@type` は単一の文字列。
- 宣言(クラスとプロパティ)は 128 個以下、全体で 65,536 UTF-16 コード単位以下。
- プロパティの `rdfs:domain` と `rdfs:range` は、それぞれ 1 つ。
- データ型プロパティの範囲は、`xsd:string` `xsd:double` `xsd:decimal` `xsd:integer` `xsd:boolean` `xsd:date` `xsd:dateTime`。

既定では、2 記事の内容に合わせた 7 クラス、7 関係、1 属性の小さなオントロジーを使います。外部 JSON-LD を指定した場合は、1.6・1.7 で検証・変換済みの内容を選びます。小さく絞るほど、抽出が安定し、費用も抑えられます。

> 出典: [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md) / [SearchServiceAttachKnowledgeSourceGraphOntology.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceAttachKnowledgeSourceGraphOntology.json) / [JSON-LD 1.1(W3C)](https://www.w3.org/TR/json-ld11/) / [OWL 2 Web Ontology Language Document Overview(W3C)](https://www.w3.org/TR/owl2-overview/)

### 8.1 抽出に使うオントロジーを確定する

内蔵定義または検証・変換した外部 JSON-LD を選び、宣言数とサイズを確認します。

In [ ]:
if RUN_BUILD:
    graph_nodes = [{"@id": f"entity:{r['class']}", "@type": "owl:Class", "rdfs:comment": r["comment"]} for _, r in classes.iterrows()]
    graph_nodes += [{"@id": f"entity:{r.predicate}", "@type": "owl:ObjectProperty", "rdfs:domain": {"@id": f"entity:{r.domain}"},
                     "rdfs:range": {"@id": f"entity:{r.range}"}, "rdfs:comment": r.comment} for r in relations.itertuples()]
    graph_nodes += [{"@id": f"entity:{r.predicate}", "@type": "owl:DatatypeProperty", "rdfs:domain": {"@id": f"entity:{r.domain}"},
                     "rdfs:range": {"@id": f"xsd:{r.xsd_range}"}, "rdfs:comment": r.comment} for r in datatype_properties.itertuples()]

    ontology = {
        "@context": {"owl": "http://www.w3.org/2002/07/owl#", "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
                     "xsd": "http://www.w3.org/2001/XMLSchema#", "entity": "https://schema.search.azure.com/graph-extraction#"},
        "@graph": graph_nodes,
    }
    ontology_jsonld = json.dumps(ontology, ensure_ascii=False, separators=(",", ":"))
    if ONTOLOGY_PATH is not None:
        ontology_jsonld = search_ontology_text
        ontology = json.loads(ontology_jsonld)
        graph_nodes = ontology["@graph"]
    search_ontology_hash = hashlib.sha256(ontology_jsonld.encode("utf-8")).hexdigest()
    units = len(ontology_jsonld.encode("utf-16-le")) // 2
    if len(graph_nodes) > 128 or units > 65536:
        raise ValueError("オントロジーの宣言数またはサイズが上限を超えています。")

    check("T7-1 宣言数が 128 以下", len(graph_nodes) <= 128, f"{len(graph_nodes)} 個")
    check("T7-2 UTF-16 コード単位が 65,536 以下", units <= 65536, f"{units}")
    display(pd.DataFrame(graph_nodes))

### 8.2 オントロジーを図にする

新規作成モードでは、テスト用スキーマのクラスをノード、関係を矢印にして描きます。共通セットアップで定義した `draw_graph` を使います(ノードの色は種類ごとです)。

In [ ]:
if RUN_BUILD:
    ontology_graph = nx.MultiDiGraph()
    for declaration in ontology["@graph"]:
        if declaration["@type"] == "owl:Class":
            ontology_graph.add_node(declaration["@id"], label=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]), kind="Class")
    for declaration in ontology["@graph"]:
        if declaration["@type"] == "owl:ObjectProperty":
            ontology_graph.add_edge(declaration["rdfs:domain"]["@id"], declaration["rdfs:range"]["@id"],
                                    predicate=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]))
    draw_graph(ontology_graph, "Foundry IQ に登録したオントロジー: クラスと関係")

## 9. T8: オントロジーの接続とトリプル抽出

新規作成モードだけで実行します。ナレッジ ソースの定義にオントロジー(`kind="inline"`)を足して、同じ名前で更新します。T5 で組み立てた `knowledge_source` オブジェクトに、オントロジーだけを足して再送します。サービスから取得した定義は、キャッシュのストレージ接続文字列が伏せられて返るため、そのまま再送すると、「エンリッチメント キャッシュのストレージ バインドは変更できない」という趣旨のエラーで拒否されます。再利用モードでは定義を更新しません。

接続しても、それだけでは抽出が始まらないことが、実環境で確認されています(プラグインも、接続後に明示的にインデクサーを実行します)。そのため、接続の直後に、生成済みのインデクサーを実行します。すでに実行中なら 409 が返るので無視します。実行すると、**キャッシュを再利用した**トリプル抽出だけが走り、グラフ インデックスにエッジが書き込まれます。**接続したオントロジーは取り除けません。**

完了の判定は、プラグインと同じです。実行中の同期がなく、直近の同期の終了時刻が、接続前より進んでいれば完了です。インデクサーが `must be retried` で終わることがありますが、再実行で回復します(`rerun=True`、最大 5 回)。書き込まれたトリプルの数は、`graphStatistics.triplesWritten` で確認します。

> 出典: [SearchServiceAttachKnowledgeSourceGraphOntology.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceAttachKnowledgeSourceGraphOntology.json) / [Deferred ontology, then attach(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-build/references/deferred-ontology.md)

In [ ]:
from azure.search.documents.indexes.models import KnowledgeSourceGraphOntology

if RUN_BUILD:
    before_attach = index_client.get_knowledge_source_status(KS_NAME).last_synchronization_state.end_time

    knowledge_source.azure_blob_parameters.graph_extraction.ontology = KnowledgeSourceGraphOntology(kind="inline", name=f"{GRAPH_NAME}-ontology", value=ontology_jsonld)
    attached = index_client.create_or_update_knowledge_source(knowledge_source)
    resources = attached.azure_blob_parameters.created_resources

    try:
        indexer_client.run_indexer(resources["indexer"])  # トリプル抽出を開始
    except HttpResponseError as e:
        if e.status_code != 409:  # 409 は、すでに実行中
            raise

    status = wait_sync(lambda current, last: current is None and last is not None and last.end_time > before_attach, rerun=True)
    state = status.as_dict()
    graph_stats = state.get("graphStatistics") or state["lastSynchronizationState"].get("graphStatistics") or {}
    chunks_after = content_client.get_document_count()
    live = index_client.get_knowledge_source(KS_NAME).azure_blob_parameters.graph_extraction

    check("T8-1 トリプルが書かれる", graph_stats.get("triplesWritten", 0) > 0, graph_stats)
    check("T8-2 チャンク数が変わらない(チャンク化とベクトル化をやり直していない)", chunks_after == chunks_before, f"接続前 {chunks_before} / 接続後 {chunks_after}")
    check("T8-3 ナレッジ ソースの定義にオントロジーが残る", live.ontology is not None and live.ontology.kind == "inline", getattr(live.ontology, "kind", None))
    registered_ontology_hash = hashlib.sha256(live.ontology.value.encode("utf-8")).hexdigest()
    if registered_ontology_hash != search_ontology_hash:
        raise RuntimeError("登録済みオントロジーと送信した JSON-LD の SHA-256 が一致しません。")
    check("T8-4 登録済みオントロジーのハッシュが一致する", True, registered_ontology_hash)
    print("直近のインデクサー実行の失敗件数:", indexer_client.get_indexer_status(resources["indexer"]).last_result.failed_item_count)
    display(pd.DataFrame([graph_stats]))

## 10. T9: グラフ インデックス

グラフ インデックスは、**1 ドキュメント = 1 本のエッジ(主語・述語・目的語)** の通常の検索インデックスです。違いは `graph_profile` で、どのフィールドが主語、述語、目的語、種類、根拠チャンクなのかを、サービスに教えます。GQL は、この対応表をもとに、インデックスをグラフとして走査します。

**再利用時は、共通セットアップと T1 の次に、このセルから再開します。** `get_knowledge_source()` で既存の定義を取得し、`createdResources.graphIndex` と `graphExtraction.graphName` から検索先を復元します。作成時の `resources` や `knowledge_source` がカーネルに残っている必要はありません。定義やグラフ インデックスが取得できない場合は、そのエラーを表示して止まります。

`createdResources.graphIndex` は、ナレッジ ソースの作成・更新直後の応答には含まれない場合があります。新規作成モードでも、T8 の同期完了後に最新の定義を再取得します。更新応答を保持した `resources` は自動的には更新されません。

```mermaid
flowchart LR
    subgraph E1["ドキュメント 1 = エッジ 1"]
        S1["subject: 近藤寺"] -- "predicate: foundedBy" --> O1["object: 近藤智海"]
    end
    subgraph EV["根拠(来歴)"]
        CH["chunk_id: コンテンツ インデックスの uid"]
    end
    E1 -. "chunk_id / parent_id" .-> EV
```

> 出典: [Create and query an entity graph: 生成リソース名の再取得](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#create-the-graph-enabled-knowledge-source) / [GraphProfile / GraphFieldMappings(azure-search-documents 12.2.0b1)](https://pypi.org/project/azure-search-documents/12.2.0b1/) / [SearchIndexGraphDocuments.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchIndexGraphDocuments.json) / Hogan et al., [Knowledge Graphs](https://arxiv.org/abs/2003.02320), ACM Computing Surveys 54(4), 2021

In [ ]:
knowledge_source = index_client.get_knowledge_source(KS_NAME)
parameters = knowledge_source.azure_blob_parameters
if parameters is None or parameters.graph_extraction is None or not parameters.graph_extraction.graph_name:
    raise RuntimeError(f"{KS_NAME}: graphExtraction.graphName がありません。Entity Graph のナレッジ ソースを指定してください。")
resources = parameters.created_resources or {}
graph_index_name = resources.get("graphIndex")
if not graph_index_name:
    raise RuntimeError(f"{KS_NAME}: createdResources.graphIndex がありません。既存グラフの作成・同期状態と KS_NAME を確認してください。")
GRAPH_NAME = parameters.graph_extraction.graph_name
graph_index = index_client.get_index(graph_index_name)
if graph_index.graph_profile is None:
    raise RuntimeError(f"{graph_index_name}: graph_profile がありません。グラフ インデックスの定義を確認してください。")
registered_ontology = json.loads(parameters.graph_extraction.ontology.value)
registered_terms = {item["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1] for item in registered_ontology["@graph"]}
sample_terms = set(classes["class"]) | set(relations["predicate"]) | set(datatype_properties["predicate"])
RUN_SAMPLE_TESTS = sample_terms <= registered_terms
if not RUN_SAMPLE_TESTS:
    print("T10〜T12 は未実施: 内蔵テスト用の型・関係がそろわないため、15・16 章で登録内容を確認してください。")
F = graph_index.graph_profile.fields
escaped_graph_name = GRAPH_NAME.replace("'", "''")
GRAPH_FILTER = f"{F.graph_name_field} eq '{escaped_graph_name}'"

graph_client = SearchClient(SEARCH_ENDPOINT, graph_index_name, credential, api_version=API_VERSION)
selectable = [f.name for f in graph_index.fields if f.retrievable is not False]
found = graph_client.search("*", filter=GRAPH_FILTER, select=selectable, include_total_count=True, top=1000)
edge_total = found.get_count()
edge_docs = pd.DataFrame(list(found))

check("T9-1 グラフ プロファイルが edgeList", getattr(graph_index.graph_profile.kind, "value", graph_index.graph_profile.kind) == "edgeList", graph_index.graph_profile.kind)
check("T9-2 グラフにエッジがある", edge_total > 0, f"{edge_total} 本")
check("T9-3 すべてのエッジに、根拠のチャンクが付く", F.source_chunk_id_field in edge_docs and bool(edge_docs[F.source_chunk_id_field].astype(bool).all()), f"根拠なし {int((~edge_docs[F.source_chunk_id_field].astype(bool)).sum())} 本")
display(pd.DataFrame([{"役割(graph_profile)": role, "フィールド名": field} for role, field in F.as_dict().items() if field]))
edge_docs.head(8)

## 11. T10: 期待する事実(再現率)

オントロジーで定義した関係(`domain`、`predicate`、`range`)ごとに、GQL の `MATCH` を実行して、エッジを集めます。ラベル(クラス名)と関係名は、バッククォートで囲みます。GQL は、`MATCH` でパターンを書き、`RETURN` で取り出す、ISO/IEC 39075 のグラフ クエリ言語です。`SearchClient.graph()` に渡し、`filter` にグラフ名のフィルターを指定します。

次に、記事に**明記されている事実**を「期待する事実」として並べ、エッジの中に見つかった割合(**再現率**)を測ります。名前は、空白と、中黒の表記ゆれ(`·` U+00B7 と `・` U+30FB)を取り除いて比べます。LLM の抽出には揺れがあるため、合格は、再現率が `MIN_RECALL` 以上であることです。

> 出典: [GQL language guide for graph in Microsoft Fabric](https://learn.microsoft.com/fabric/graph/gql-language-guide) / [ISO/IEC 39075](https://www.iso.org/standard/76120.html) / [Graph traversal(foundry-iq-skills)](https://github.com/microsoft/azure-skills/blob/main/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/entity-graph-consume/references/graph-traversal.md) / [sample_query_graph_preview.py](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/samples/sample_query_graph_preview.py)

In [ ]:
if RUN_SAMPLE_TESTS:
    frames = []
    for rel in relations.itertuples():
        result = graph_client.graph(gql=f"MATCH (a:`{rel.domain}`)-[r:`{rel.predicate}`]->(b:`{rel.range}`) RETURN a.name AS a, type(r) AS predicate, b.name AS b", filter=GRAPH_FILTER)
        frames.append(pd.DataFrame([row.as_dict() for row in result.value], columns=["a", "predicate", "b"]).assign(a_type=rel.domain, b_type=rel.range))
    edges = pd.concat(frames, ignore_index=True)

    EXPECTED = pd.DataFrame([
        ("近藤寺", "foundedBy", ["近藤智海"]), ("近藤寺", "locatedIn", ["京都市東山区"]), ("近藤寺", "enshrines", ["百八面千手観世音菩薩"]),
        ("近藤寺", "survivedEvent", ["応仁の乱"]), ("近藤寺", "registeredAs", ["古都京都の文化財"]),
        ("花ケ咲神社", "locatedIn", ["京都市東山区"]), ("花ケ咲神社", "enshrines", ["桜花女神"]),
        ("花ケ咲神社", "survivedEvent", ["応仁の乱"]), ("花ケ咲神社", "registeredAs", ["古都京都の文化財"]),
        ("ビル・ウィンドウズ", "visited", ["舞供楼", "阿須杯庵"]),
    ], columns=["a", "predicate", "b_any"])
    EXPECTED["見つかった"] = [
        bool(((edges["predicate"] == e.predicate) & edges["a"].map(lambda x: norm(e.a) in norm(x)) & edges["b"].map(lambda x: any(norm(k) in norm(x) for k in e.b_any))).any())
        for e in EXPECTED.itertuples()
    ]
    recall = EXPECTED["見つかった"].mean()

    check("T10 期待する事実の再現率が MIN_RECALL 以上", recall >= MIN_RECALL, f"{int(EXPECTED['見つかった'].sum())} / {len(EXPECTED)} = {recall:.0%}(下限 {MIN_RECALL:.0%})")
    print("抽出されたエッジの数:", len(edges))
    EXPECTED

### 11.1 関係の種類ごとの件数と、グラフ全体の図

ノードの色は、クラス(種類)です。2 記事が、所在地、世界遺産の指定、応仁の乱のノードを共有して、つながる様子を確認できます。

In [ ]:
if RUN_SAMPLE_TESTS:
    display(edges.groupby(["a_type", "predicate", "b_type"]).size().rename("エッジ数").reset_index())

    instance_graph = nx.DiGraph()
    for e in edges.itertuples():
        instance_graph.add_node(e.a, type=e.a_type)
        instance_graph.add_node(e.b, type=e.b_type)
        instance_graph.add_edge(e.a, e.b, predicate=e.predicate)
    print("ノード数:", instance_graph.number_of_nodes(), "/ エッジ数:", instance_graph.number_of_edges())
    draw_graph(instance_graph, f"ナレッジ グラフ '{GRAPH_NAME}'", figsize=(15, 11))

## 12. T11: 文書をまたぐ共有ノード

2 つの記事が、同じ事実(所在地、焼失を免れた出来事、世界遺産の指定)を書いています。グラフでは、**同じエンティティが 1 つのノードになり、両方の寺社がそこにつながる**はずです。共通のノードを 2 方向からたどるパターンで探します。`a.name < b.name` は、同じ組が順序を入れ替えて 2 回出るのを避ける条件です。通常の検索では、2 つの文書から該当文を拾って突き合わせる必要がありますが、グラフでは 1 つのパターンで表せます。

In [ ]:
if RUN_SAMPLE_TESTS:
    shared = {}
    for predicate, cls in [("locatedIn", "Place"), ("survivedEvent", "Event"), ("registeredAs", "Designation")]:
        gql = (f"MATCH (a:ReligiousSite)-[:{predicate}]->(n:{cls})<-[:{predicate}]-(b:ReligiousSite) "
               "WHERE a.name < b.name RETURN a.name AS site_a, b.name AS site_b, n.name AS shared")
        shared[predicate] = rows(run_gql(gql))
        check(f"T11 {predicate}: 2 つの寺社が同じノードにつながる", len(shared[predicate]) >= 1, shared[predicate].to_dict("records")[:1])
    pd.concat({k: v for k, v in shared.items()})

## 13. T12: マルチホップと、来歴(根拠のチャンクまで遡る)

「ビル・ゲイツが訪れた庭を持つ寺を創建したのは誰か」を、3 ホップのパターンで解きます(`visited`、`hasFacility`、`foundedBy`)。質問の型は、エンティティ名を固定せず、パターンで書きます。抽出されたエンティティ名の表記が、実行ごとに揺れても、結果が空になりにくくするためです。

エッジには、抽出元のチャンクの ID(`chunk_id`)が記録されています。クエリに `provenance` を指定すると、各行について、根拠のエッジと、その元になったチャンクの本文を返します。

- `kind="ids"`: 根拠のエッジとチャンクの ID だけを返します。
- `kind="hydrate"`: チャンクのドキュメントも取得します(`select` で返すフィールドを指定します)。

グラフの答えを、原文で確かめるために使います。

> 出典: [KnowledgeBaseRetrieveGraph.json / SearchIndexGraphDocuments.json](https://github.com/Azure/azure-rest-api-specs/tree/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples) / [sample_query_graph_preview.py](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/search/azure-search-documents/samples/sample_query_graph_preview.py)

In [ ]:
if RUN_SAMPLE_TESTS:
    MULTI_HOP = (
        "MATCH (p:Person)-[:visited]->(f:Facility)<-[:hasFacility]-(s:ReligiousSite)-[:foundedBy]->(g:Person) "
        "RETURN p.name AS visitor, f.name AS facility, s.name AS site, g.name AS founder"
    )
    result = run_gql(MULTI_HOP, provenance=GraphQueryProvenanceParameters(
        kind="hydrate", max_evidence_per_row=3, hydrate_documents=GraphQueryProvenanceHydrateDocumentsParameters(select="uid,snippet")))
    hop = rows(result)

    evidence = []
    for row in result.value:
        d = row.as_dict()
        for ev in d["@search.graphProvenance"]["evidence"]:
            doc = result.provenance_documents.get(ev["chunkId"], {})
            evidence.append({"facility": d["facility"], "site": d["site"], "founder": d["founder"], "chunk_id": ev["chunkId"], "根拠のチャンク": (doc.get("snippet") or "")[:140]})
    evidence_df = pd.DataFrame(evidence, columns=["facility", "site", "founder", "chunk_id", "根拠のチャンク"])

    check("T12-1 3 ホップの結果に、創建者(近藤智海)が含まれる", len(hop) >= 1 and hop["founder"].map(lambda x: "近藤智海" in norm(x)).any(), hop.to_dict("records")[:1])
    check("T12-2 根拠のチャンクの本文を取得できる", len(evidence_df) >= 1 and bool(evidence_df["根拠のチャンク"].str.len().gt(0).all()), f"根拠 {len(evidence_df)} 件")

    path_graph = nx.DiGraph()
    for r in hop.itertuples():
        path_graph.add_node(r.visitor, type="Person")
        path_graph.add_node(r.founder, type="Person")
        path_graph.add_node(r.facility, type="Facility")
        path_graph.add_node(r.site, type="ReligiousSite")
        path_graph.add_edge(r.visitor, r.facility, predicate="visited")
        path_graph.add_edge(r.site, r.facility, predicate="hasFacility")
        path_graph.add_edge(r.site, r.founder, predicate="foundedBy")
    if len(hop):
        draw_graph(path_graph, "マルチホップの検索結果", figsize=(10, 6))
    evidence_df

## 14. T13: ナレッジ ベース(任意)

ナレッジ ベースは、ナレッジ ソースを束ね、質問に対して LLM が検索計画を立て、グラフ検索(GQL を生成して実行)と本文検索を使い分けて、引用つきの回答を合成します。**LLM を呼ぶため、`RUN_KB_TEST = True` のときだけ実行します**。次を確かめます。

- 回答の文章と、参照(`[ref_id:N]` の指す先)が返る。
- 実行ログに、グラフ検索(`graphSearch`)が含まれる。

グラフ検索が空だった場合、ナレッジ ベースは、本文検索の結果で答えようとします。質問の中のエンティティ名の表記が、グラフのノード名と一致しないと、`EmptyResults` になることがあります(記事中の「ビル·ゲイツ」の中黒は U+00B7 で、質問に書く「ビル・ゲイツ」の U+30FB とは別の文字です)。そのため、ここでは、表記ゆれの影響を受けにくい質問を使います。

> 出典: [Query a knowledge base using the retrieve action](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve) / [KnowledgeBaseRetrieveGraph.json](https://github.com/Azure/azure-rest-api-specs/blob/main/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/KnowledgeBaseRetrieveGraph.json)

In [ ]:
from azure.search.documents.indexes.models import AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel, KnowledgeSourceReference
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    AgentRuntimeGraphHydrateDocumentsParams,
    AgentRuntimeGraphParams,
    AgentRuntimeGraphProvenanceParams,
    AzureBlobKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

if RUN_KB_TEST:
    chat_params = AzureOpenAIVectorizerParameters(resource_url=AOAI_ENDPOINT, deployment_name=CHAT_DEPLOYMENT, model_name=CHAT_MODEL)
    index_client.create_or_update_knowledge_base(KnowledgeBase(
        name=KB_NAME,
        description="ハナペディアの寺社記事(近藤寺、花ケ咲神社)から構築したナレッジ グラフと本文チャンク。",
        knowledge_sources=[KnowledgeSourceReference(name=KS_NAME)],
        models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=chat_params)],
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        retrieval_instructions=("Use *_graph_search tools for structured questions about entities, their properties, and relationships, including finding entities that meet "
                                "specific criteria or tracing connections. Use relevance-based search tools to find information by topic, meaning, or wording. "
                                "Express graph requests as natural-language questions describing the entities, relationships, and criteria sought."),
        answer_instructions=("Answer only from the retrieved graph rows and passages. Cite every claim with its [ref_id:N]. Do not add facts that the retrieved results do not state."),
    ))
    kb_client = KnowledgeBaseRetrievalClient(SEARCH_ENDPOINT, credential, knowledge_base_name=KB_NAME)
    answer = kb_client.retrieve(KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text="近藤寺と花ケ咲神社が共通して焼失を免れた出来事は何ですか?")])],
        include_activity=True, output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS, max_runtime_in_seconds=120,
        knowledge_source_params=[AzureBlobKnowledgeSourceParams(
            knowledge_source_name=KS_NAME, include_references=True, include_reference_source_data=True,
            graph_parameters=AgentRuntimeGraphParams(include_query=True, provenance=AgentRuntimeGraphProvenanceParams(
                kind="hydrate", hydrate_documents=AgentRuntimeGraphHydrateDocumentsParams(top=10))))],
    ))
    activity = pd.DataFrame([{"種類": a.as_dict()["type"], "結果": (a.as_dict().get("graphSearchArguments") or {}).get("outcome"),
                              "GQL": (a.as_dict().get("graphSearchArguments") or {}).get("gql")} for a in answer.activity])
    check("T13-1 回答と参照が返る", bool(answer.response[0].content[0].text) and len(answer.references) >= 1, f"参照 {len(answer.references)} 件")
    check("T13-2 実行ログにグラフ検索が含まれる", (activity["種類"] == "graphSearch").any(), activity["種類"].tolist())
    print("回答:", answer.response[0].content[0].text)
    display(activity)

## 15. Foundry IQ の登録内容を読み戻して可視化する

T5・T8 で作成・更新したナレッジ ソース、または `KS_NAME` の既存ソースをそのまま読み取ります。ここでは新しいソースの作成、再取り込み、モデル呼び出しを行いません。新しいカーネルでも **1.2 の設定（`RUN_BUILD = False`）→ 1.5 の描画ヘルパー → T1 → この章** で再開できます。

図はノートブック内の設計データではなく、Foundry IQ から再取得した登録内容に基づきます。最大 `MAX_DISPLAY_EDGES` 文書の部分グラフを表示します。

> 出典: [Entity Graph の作成と直接検索](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph)

### 15.1 登録済みオントロジー

ナレッジ ソースの最新定義から JSON-LD・グラフ名・生成インデックス名を取得します。クラスをノード、オブジェクト プロパティを矢印にした設計図です。表示するハッシュは現在の登録内容の識別用です。送信内容との一致は T8 で検証します。

> 出典: [Knowledge Sources - Get](https://learn.microsoft.com/rest/api/searchservice/knowledge-sources/get?view=rest-searchservice-2026-11-01-preview) / [OWL 2 Primer](https://www.w3.org/TR/owl2-primer/)

In [ ]:
ENTITY_KS_NAME = KS_NAME
registered_source = index_client.get_knowledge_source(ENTITY_KS_NAME)
parameters = registered_source.azure_blob_parameters
if parameters is None or parameters.graph_extraction is None or parameters.graph_extraction.ontology is None:
    raise ValueError("登録済みの graphExtraction とオントロジーが必要です。")
ENTITY_GRAPH_NAME = parameters.graph_extraction.graph_name
resources = parameters.created_resources or {}
registered_ontology_text = parameters.graph_extraction.ontology.value
registered_ontology_hash = hashlib.sha256(registered_ontology_text.encode("utf-8")).hexdigest()
registered_ontology = json.loads(registered_ontology_text)
print("登録済みオントロジーの SHA-256:", registered_ontology_hash)
display(pd.DataFrame([{"ナレッジ ソース": ENTITY_KS_NAME, "グラフ名": ENTITY_GRAPH_NAME, "グラフ インデックス": resources.get("graphIndex"), "コンテンツ インデックス": resources.get("index")}]))

In [ ]:
ontology_graph = nx.MultiDiGraph()
for declaration in registered_ontology["@graph"]:
    if declaration["@type"] == "owl:Class":
        ontology_graph.add_node(declaration["@id"], label=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]), kind="Class")
for declaration in registered_ontology["@graph"]:
    if declaration["@type"] == "owl:ObjectProperty":
        ontology_graph.add_edge(declaration["rdfs:domain"]["@id"], declaration["rdfs:range"]["@id"],
                                predicate=declaration.get("rdfs:label", declaration["@id"].rsplit("#", 1)[-1].rsplit(":", 1)[-1]))
draw_graph(ontology_graph, "Foundry IQ に登録したオントロジー: クラスと関係")

## 15.2 抽出済みエンティティと関係の可視化

15.1 で取得した `createdResources.graphIndex` を使い、待機セルは実行しません。グラフ インデックスがまだない場合は明示的に止まります。抽出の進行後に 15.1 の定義取得から再実行してください。

フィールド名は、登録済みインデックスの `graph_profile.fields` を使います。図は `object_id` があるエンティティ間の関係に限定します。文字列・数値などの属性は、取得した文書の表で確認できます。図は取得上限内の部分グラフであり、全グラフを常に表示するものではありません。抽出中に実行すると、その時点までに登録されたデータが表示されます。

> 出典: [Entity Graph のインデックスと来歴](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
entity_index_name = resources.get("graphIndex")
if not entity_index_name:
    raise RuntimeError("ナレッジ ソースの createdResources に graphIndex がありません。抽出状態を確認し、15.1 の定義取得から再実行してください。")
entity_index = index_client.get_index(entity_index_name)
F = entity_index.graph_profile.fields
entity_client = SearchClient(SEARCH_ENDPOINT, entity_index_name, credential, api_version=API_VERSION)
entity_select = [field.name for field in entity_index.fields if field.retrievable is not False]
escaped_graph_name = ENTITY_GRAPH_NAME.replace("'", "''")


entity_found = entity_client.search("*", filter=f"{F.graph_name_field} eq '{escaped_graph_name}'", select=entity_select,
                                    include_total_count=True, top=MAX_DISPLAY_EDGES)
entity_total = entity_found.get_count()
entity_frame = pd.DataFrame(list(entity_found))
if entity_frame.empty:
    raise RuntimeError("指定したグラフにエッジがありません。抽出状態、オントロジー、インデクサーの結果を確認してください。")
entity_relations = entity_frame[entity_frame[F.object_id_field].fillna("").ne("")]
if entity_relations.empty:
    raise RuntimeError("取得範囲にエンティティ間の関係がありません。上限やオントロジーを確認してください。")
print(f"グラフ内 {entity_total} 文書 / 取得 {len(entity_frame)} 文書 / 図の関係 {len(entity_relations)} 本")
display(entity_frame[[F.subject_id_display_name_field, F.predicate_field, F.object_id_display_name_field, F.object_string_field, F.source_chunk_id_field]].head(12))

In [ ]:
entity_graph = nx.MultiDiGraph()
for row in entity_relations.to_dict("records"):
    entity_graph.add_node(row[F.subject_id_field], label=row[F.subject_id_display_name_field], kind=row[F.subject_type_field])
    entity_graph.add_node(row[F.object_id_field], label=row[F.object_id_display_name_field], kind=row[F.object_type_field])
    entity_graph.add_edge(row[F.subject_id_field], row[F.object_id_field], predicate=row[F.predicate_field])
draw_graph(entity_graph, f"Foundry IQ の実データ: {entity_index_name}")

### 関係の根拠を 1 件確認する

抽出結果のチャンク ID を、同じナレッジ ソースのコンテンツ インデックスに結び付けます。トリプルがあるだけでなく、どの本文を根拠にしたかを確認します。可視化の配置やモデルの出力だけから、記事にない事実を補いません。

> 出典: [グラフの来歴と元チャンク](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
edge = entity_relations.iloc[0]
chunk_id = edge[F.source_chunk_id_field]
if not isinstance(chunk_id, str) or not chunk_id:
    raise RuntimeError("選んだ関係に根拠チャンク ID がありません。")
content_client = SearchClient(SEARCH_ENDPOINT, resources["index"], credential, api_version=API_VERSION)
evidence = content_client.get_document(chunk_id, selected_fields=["uid", "blob_url", "snippet"])
display(pd.DataFrame([{"主語": edge[F.subject_id_display_name_field], "関係": edge[F.predicate_field], "目的語": edge[F.object_id_display_name_field]}]))
display(pd.DataFrame([evidence]))

In [ ]:
from azure.search.documents.models import (
    GraphQueryResult,
    GraphQueryProvenanceParameters,
    GraphQueryProvenanceHydrateDocumentsParameters,
)


def display_gql(result: GraphQueryResult) -> None:
    print("返却行数:", len(result.value))
    display(pd.json_normalize(result.as_dict()["value"]))
    if result.diagnostics:
        print("診断（警告・エラー・実行情報）:")
        display(pd.DataFrame([item.as_dict() for item in result.diagnostics]))
    if result.statistics:
        print("実行統計:")
        display(pd.DataFrame([result.statistics.as_dict()]))

## 16 GQL による探索・集計・根拠確認

15 章末尾の GQL 表示処理と、15.1 の登録済み定義の取得セルを実行してから進みます。15.1 の描画セル、15.2 の文書検索・描画・根拠取得、他のノートブックは不要です。型と関係 → 名称 → 2 ホップの経路 → 所在地別の集計 → 根拠本文の順に確認します。

> 出典: [Entity Graph の直接 GQL クエリ](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

### 16.0 グラフ クライアントの準備

15.1 で最新のナレッジ ソース定義を取得してから実行します。描画セルや T10〜T12 は不要です。以下は内蔵オントロジーの `ReligiousSite`、`Deity`、`Facility`、`Place` と関係名を使います。外部 JSON-LD の場合は 15.1 の定義に合わせてクエリを変更してください。GQL はモデルを呼び出しません。

> 出典: [GQL の直接実行](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
gql_entity_index_name = resources.get("graphIndex")
if not gql_entity_index_name:
    raise RuntimeError("createdResources に graphIndex がありません。抽出状態を確認し、15.1 の定義取得から再実行してください。")
gql_entity_index = index_client.get_index(gql_entity_index_name)
if gql_entity_index.graph_profile is None or gql_entity_index.graph_profile.kind != "edgeList":
    raise ValueError("GQL には edgeList の graphProfile が必要です。")
gql_graph_field = gql_entity_index.graph_profile.fields.graph_name_field
gql_graph_name = ENTITY_GRAPH_NAME.replace("'", "''")
entity_gql_filter = f"{gql_graph_field} eq '{gql_graph_name}'"
entity_gql_client = SearchClient(SEARCH_ENDPOINT, gql_entity_index_name, credential, api_version=API_VERSION)

### 16.1 型と関係で絞り込む

`ReligiousSite` から `Deity` へ向かう `enshrines` を取得します。型ラベルと述語は別の条件です。

> 出典: [型と関係を指定する GQL](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
entity_relations_gql = """
MATCH (site:ReligiousSite)-[edge]->(deity:Deity)
WHERE edge.predicate = 'enshrines'
RETURN site.id AS site_id, site.name AS site, deity.name AS deity
ORDER BY site, deity
LIMIT 10
"""
entity_relations_result = entity_gql_client.graph(
    gql=entity_relations_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_relations_result)

### 16.2. 名称を起点に探索する

「近藤寺」という表示名を持つノードから外向きの関係を取得します。結果には ID も含めるため、同名の別ノードを区別できます。特定の ID に絞る場合は、`subject.name = '近藤寺'` を `subject.id = '取得した ID'` に置き換えます。複数の値が返った場合は 16.5 の根拠と照合してください。

> 出典: [名称による絞り込み](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [ID による絞り込みの例](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
entity_name_gql = """
MATCH (subject)-[edge]->(object)
WHERE subject.name = '近藤寺'
RETURN subject.id AS subject_id, subject.name AS subject_name,
       edge.predicate AS relation, object.id AS object_id, object.name AS object_name
ORDER BY relation, object_name
LIMIT 10
"""
entity_name_result = entity_gql_client.graph(
    gql=entity_name_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_name_result)

### 16.3 施設と訪問者を 2 ホップでたどる

`宗教施設 → hasFacility → 境内施設 ← visited ← 人物` の経路を取得します。中間の施設を共有する 1 クエリです。

> 出典: [複数ホップの GQL](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql)

In [ ]:
entity_path_gql = """
MATCH (site)-[first]->(facility)<-[second]-(person)
WHERE first.predicate = 'hasFacility' AND second.predicate = 'visited'
RETURN site.name AS site, facility.name AS facility, person.name AS visitor
ORDER BY site, facility, visitor
LIMIT 10
"""
entity_path_result = entity_gql_client.graph(
    gql=entity_path_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_path_result)

### 16.4 所在地ごとの関係数を集計する

`locatedIn` の経路を場所の ID・表示名ごとに集計します。`relation_count` は一致した経路の行数で、根拠文書数や重複排除済みの施設数とは限りません。

> 出典: [Documents - Graph](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
entity_count_gql = """
MATCH (site)-[edge]->(place)
WHERE edge.predicate = 'locatedIn'
RETURN place.id AS place_id, place.name AS place, count(*) AS relation_count
GROUP BY place.id, place.name
ORDER BY relation_count DESC, place
LIMIT 10
"""
entity_count_result = entity_gql_client.graph(
    gql=entity_count_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
)
display_gql(entity_count_result)

### 16.5. GQL の結果と根拠の本文を一緒に取得する

`provenance.kind = hydrate` にすると、各結果行の `@search.graphProvenance` にエッジ・チャンク・親文書の ID が付き、本文は応答全体の `@search.graphProvenanceDocuments` にまとめて返ります。読み込み先は `graphProfile.provenanceIndexName` で指定されたコンテンツ インデックスです。Python 側で 15.2 の検索結果を使い回すわけではありません。

`max_evidence_per_row = 3` は 1 行あたりの取得根拠数の上限です。`evidenceCount.relation` が `exact` なら件数は完全で、`lowerBound` ならさらに根拠がある可能性があります。Topic Graph にも GQL は使えますが、このリリースの Topic Graph インデックスには `provenanceIndexName` がないため、ここで示す本文の自動取得は管理された Entity Graph に対して実行します。

> 出典: [GQL と根拠文書の自動取得](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-directly-with-gql) / [Documents - Graph の provenance](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
if not gql_entity_index.graph_profile.provenance_index_name:
    raise ValueError("本文の自動取得には graphProfile.provenanceIndexName が必要です。")
entity_evidence_gql = """
MATCH (site)-[edge]->(deity)
WHERE site.name = '近藤寺' AND edge.predicate = 'enshrines'
RETURN site.name AS site, deity.name AS deity
ORDER BY deity
LIMIT 10
"""
entity_evidence_result = entity_gql_client.graph(
    gql=entity_evidence_gql, filter=entity_gql_filter,
    include_statistics=True, include_diagnostics=True,
    provenance=GraphQueryProvenanceParameters(
        kind="hydrate", max_evidence_per_row=3,
        hydrate_documents=GraphQueryProvenanceHydrateDocumentsParameters(select="uid,blob_url,snippet"),
    ),
)
display_gql(entity_evidence_result)
display(pd.DataFrame.from_dict(
    entity_evidence_result.as_dict()["@search.graphProvenanceDocuments"], orient="index",
).rename_axis("根拠チャンク ID"))

In [ ]:
check("T14-1 登録済みオントロジーを取得できる", bool(registered_ontology["@graph"]), registered_ontology_hash)
check("T14-2 登録済みの実体間の関係を取得できる", not entity_relations.empty, f"{len(entity_relations)} 本を描画")
check("T14-3 根拠本文を取得できる", bool(evidence.get("snippet")), edge[F.source_chunk_id_field])
display(pd.DataFrame([{"ナレッジ ソース": KS_NAME, "グラフ インデックス": entity_index_name,
                       "グラフ内の文書数": entity_total, "取得数": len(entity_frame)}]))

## 17. 結果の一覧

実行したテストだけを集計し、PASS と FAIL の件数と、FAIL の詳細を表示します。再利用モードでは T2〜T8 を合格扱いにはせず、一覧に含めません。LLM の抽出に揺れがあるため、T10 と T12 が FAIL のときは、再実行して再現率や抽出されたエッジを見比べてください。表記ゆれ(中黒、「ケ」と「ヶ」など)で、ノードが分かれていることもあります。

In [ ]:
summary = pd.DataFrame(results)
print(summary["結果"].value_counts().to_dict())
summary.style.apply(lambda row: ["background-color: #ffd6d6" if row["結果"] == "FAIL" else "" for _ in row], axis=1)

## 18. 後片付け(任意)

**新規作成モードで `RUN_CLEANUP = True` にした場合だけ**、ナレッジ ベースとナレッジ ソースを削除します。再利用モードで削除を指定した場合はエラーで止め、既存リソースを保護します。ナレッジ ソースを削除すると、自動生成されたデータ ソース、インデクサー、スキルセット、コンテンツ インデックス、グラフ インデックスも、一緒に削除されます。

次のものは削除されないので、不要なら Storage 側で個別に削除してください。

- キャッシュ用ストレージの `ms-az-search-indexercache-` で始まるコンテナーとテーブル
- ソース用ストレージの Blob コンテナー(`ft-entity`)と、アップロードしたファイル

In [ ]:
if RUN_CLEANUP and not RUN_BUILD:
    raise ValueError("再利用モードでは既存リソースを削除しません。RUN_CLEANUP = False にしてください。")

if RUN_CLEANUP:
    for delete, name in ((index_client.delete_knowledge_base, KB_NAME), (index_client.delete_knowledge_source, KS_NAME)):
        try:
            delete(name)
            print("削除しました:", name)
        except ResourceNotFoundError:
            pass